<a href="https://colab.research.google.com/github/MarcelDiaz/HW2-BigData/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Práctica HW2 Big Data. \
Marcel, Pau, Ane

#**Imports**

In [1]:
from pyspark.sql.functions import col, sum, year, round, hour, dayofweek, month, when

#**Introducing the Data: Yellow Taxis from January and February 2026**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName('NYC_Taxi_Analysis') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os

os.listdir('/content/drive/MyDrive/')

['DBH 1 2 3 ',
 'My Video copy.mov',
 'eb-garamond.zip',
 'Gazta tarta.gdoc',
 'Classroom',
 'My Movie 2.iMovieMobile',
 'DIVERGENT.docx',
 'Gaur euria ari du.docx',
 'hirugarren ariketa.docx',
 'Ingles (1).gdoc',
 'SEGURTASUN_INFORMATIKOA.pdf',
 'SEGURTASUN_INFORMATIKOA.gdoc',
 'BATX',
 'Persiarren manifestua.gdoc',
 'testuakdenakkomentatuta.docx',
 'Ingles.gdoc',
 'Joifjsejf.gdoc',
 'EGANTZAK PLANTILLAK.gdoc',
 'Hoja de cálculo sin título (2).gsheet',
 'Documento sin título (9).gdoc',
 'Práctica - 2.gsheet',
 'KUADRILAKO DNIXAK ETAB..gdoc',
 'test-cleaver-disc_a22681a0238e6c5e3e066ea8318eacd9 (1).xlsx',
 'test-cleaver-disc_a22681a0238e6c5e3e066ea8318eacd9.xlsx',
 'Edos trabajo.gslides',
 'INGLES ANALISISVectorCalculus.pdf',
 'GEMiF',
 'ESPAÑOL.Calculo.Vectorial._3Ed_.pdf',
 '2be74198-5cfb-472c-a3f1-f71f4d69a1f3.JPG',
 'Jam sin título.pdf',
 'drive-download-20230628T133643Z-001.zip',
 'JAPON 2023',
 'bfk-ajek-nic - 11 nov 2023.pdf',
 'Hoja de cálculo sin título (1).gsheet',
 '

In [5]:
# Rutas en Google Drive
path_jan = '/content/drive/MyDrive/yellow_tripdata_2026-01.parquet'
path_feb = '/content/drive/MyDrive/yellow_tripdata_2026-02.parquet'

# Leer datos
df_jan = spark.read.parquet(path_jan)
df_feb = spark.read.parquet(path_feb)

# Unificar
df = df_jan.union(df_feb)

# Inspección
df.printSchema()
print(f'Total de registros: {df.count()}')

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Total de registros: 7124755


#**Data Preprocessing**

Identificar y tratar registros problemáticos, como valores nulos en campos clave.

In [6]:
# Conteo inicial para documentar
total_inicial = df.count()
print(f"Total de registros iniciales: {total_inicial}")

# Calcular la cantidad de valores nulos por cada columna
nulos_por_columna = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

print("Cantidad de valores nulos por columna:")
nulos_por_columna.show(vertical=True)

Total de registros iniciales: 7124755
Cantidad de valores nulos por columna:
-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 2111375 
 trip_distance         | 0       
 RatecodeID            | 2111375 
 store_and_fwd_flag    | 2111375 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 2111375 
 Airport_fee           | 2111375 
 cbd_congestion_fee    | 0       



In [7]:
# Opción A (Recomendada): Eliminar registros sin datos de tarifa, distancia o pasajeros
columnas_importantes = ["passenger_count", "trip_distance", "fare_amount"]
df_limpio = df.dropna(subset=columnas_importantes)

# Opción B (Alternativa): Imputar (rellenar) los pasajeros nulos con 1 (el mínimo lógico)
# df_limpio = df_limpio.fillna({"passenger_count": 1})

# Conteo final para ver cuántos registros perdimos
total_sin_nulos = df_limpio.count()
registros_eliminados = total_inicial - total_sin_nulos

print(f"Total de registros tras limpiar nulos: {total_sin_nulos}")
print(f"Registros eliminados por falta de datos: {registros_eliminados} ({(registros_eliminados/total_inicial)*100:.2f}%)")

Total de registros tras limpiar nulos: 5013380
Registros eliminados por falta de datos: 2111375 (29.63%)


Eliminar o justificar registros con distancia, duración o tarifa cero o negativa.

In [8]:
# Guardamos el conteo antes de este paso para el informe
total_antes_filtros = df_limpio.count()

# 1. Filtro de distancia: Un viaje debe tener una distancia mayor a 0
df_limpio = df_limpio.filter(col("trip_distance") > 0)

# 2. Filtro de tarifas: Eliminamos tarifas en cero o negativas (suelen ser cancelaciones o errores)
df_limpio = df_limpio.filter((col("fare_amount") > 0) & (col("total_amount") > 0))

# 3. Filtro de tiempo (Duración): La fecha/hora de bajada debe ser estrictamente mayor a la de subida.
# Esto previene duraciones cero o negativas antes de que crees la variable en el Punto 3.
df_limpio = df_limpio.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

# Conteo posterior y cálculo de eliminados
total_despues_filtros = df_limpio.count()
registros_filtrados = total_antes_filtros - total_despues_filtros

print(f"Registros eliminados por ceros/negativos: {registros_filtrados}")
print(f"Total de registros válidos hasta ahora: {total_despues_filtros}")

Registros eliminados por ceros/negativos: 209369
Total de registros válidos hasta ahora: 4804011


Inspeccionar cantidades irreales de pasajeros y tiempos inconsistentes de
recogida/llegada.

In [9]:
# Guardamos el conteo antes de este filtro
total_antes_pasajeros = df_limpio.count()

# Filtramos viajes con 0 pasajeros o con más de 6 (límite realista)
df_limpio = df_limpio.filter((col("passenger_count") > 0) & (col("passenger_count") <= 6))

registros_eliminados_pasajeros = total_antes_pasajeros - df_limpio.count()
print(f"Registros eliminados por cantidad irreal de pasajeros: {registros_eliminados_pasajeros}")

Registros eliminados por cantidad irreal de pasajeros: 26375


In [10]:
# Asumiendo que elegiste el año 2024 para tu análisis
año_analisis = 2026

total_antes_tiempo = df_limpio.count()

# 1. Validar que el año de recogida sea el correcto
df_limpio = df_limpio.filter(year(col("tpep_pickup_datetime")) == año_analisis)

# 2. (Opcional pero recomendado) Validar que el año de bajada no sea un disparate
# Permitimos que la bajada sea en el mismo año, o a lo sumo en los primeros días del año siguiente
df_limpio = df_limpio.filter(
    (year(col("tpep_dropoff_datetime")) == año_analisis) |
    (year(col("tpep_dropoff_datetime")) == año_analisis + 1)
)

registros_eliminados_tiempo = total_antes_tiempo - df_limpio.count()
print(f"Registros eliminados por años fuera de rango temporal: {registros_eliminados_tiempo}")

# --- RESUMEN FINAL DE LIMPIEZA ---
total_final_limpio = df_limpio.count()
print(f"--- LIMPIEZA COMPLETADA ---")
print(f"Registros válidos finales: {total_final_limpio}")

Registros eliminados por años fuera de rango temporal: 5
--- LIMPIEZA COMPLETADA ---
Registros válidos finales: 4777631


#**Engineering of Characteristics**

In [11]:
from pyspark.sql.functions import unix_timestamp

# Calculamos la diferencia en segundos, dividimos por 60 y redondeamos a 2 decimales
df_features = df_limpio.withColumn(
    "trip_duration_minutes",
    round((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60.0, 2)
)

# Mostramos un par de registros para validar que el cálculo tiene sentido
df_features.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(5, truncate=False)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
|2026-01-01 00:54:04 |2026-01-01 00:59:37  |5.55                 |
|2026-01-01 00:15:22 |2026-01-01 00:58:10  |42.8                 |
|2026-01-01 00:47:11 |2026-01-01 01:00:47  |13.6                 |
|2026-01-01 00:17:54 |2026-01-01 00:28:32  |10.63                |
|2026-01-01 00:34:14 |2026-01-01 01:11:58  |37.73                |
+--------------------+---------------------+---------------------+
only showing top 5 rows


In [12]:
df_features = df_features \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime")))

# Verificamos los resultados
df_features.select(
    "tpep_pickup_datetime",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month"
).show(5)

+--------------------+-----------+------------------+------------+
|tpep_pickup_datetime|pickup_hour|pickup_day_of_week|pickup_month|
+--------------------+-----------+------------------+------------+
| 2026-01-01 00:54:04|          0|                 5|           1|
| 2026-01-01 00:15:22|          0|                 5|           1|
| 2026-01-01 00:47:11|          0|                 5|           1|
| 2026-01-01 00:17:54|          0|                 5|           1|
| 2026-01-01 00:34:14|          0|                 5|           1|
+--------------------+-----------+------------------+------------+
only showing top 5 rows


In [13]:
# Calculamos la velocidad en millas por hora (mph)
# Fórmula: (Distancia / Minutos) * 60
# Usamos 'when' para evitar divisiones por cero en caso de que algún viaje haya durado 0 minutos exactos
df_features = df_features.withColumn(
    "avg_speed_mph",
    when(col("trip_duration_minutes") > 0,
         round((col("trip_distance") / col("trip_duration_minutes")) * 60, 2))
    .otherwise(0) # Si el tiempo es 0 o negativo, asignamos 0 a la velocidad
)

# Verificamos los resultados
df_features.select(
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph"
).show(5)

+-------------+---------------------+-------------+
|trip_distance|trip_duration_minutes|avg_speed_mph|
+-------------+---------------------+-------------+
|         0.97|                 5.55|        10.49|
|         5.58|                 42.8|         7.82|
|         2.33|                 13.6|        10.28|
|          1.3|                10.63|         7.34|
|         5.34|                37.73|         8.49|
+-------------+---------------------+-------------+
only showing top 5 rows


In [14]:
# Factor de conversión: 1 milla = 1.60934 kilómetros
FACTOR_KM = 1.60934

# 1. Creamos la columna de distancia en kilómetros
df_features = df_features.withColumn(
    "trip_distance_km",
    round(col("trip_distance") * FACTOR_KM, 2)
)

# 2. Calculamos la tarifa por kilómetro (Eficiencia)
# Usamos fare_amount para medir estrictamente el coste del viaje (sin propinas ni peajes)
df_features = df_features.withColumn(
    "fare_per_km",
    when(col("trip_distance_km") > 0,
         round(col("fare_amount") / col("trip_distance_km"), 2))
    .otherwise(0)
)

# Verificamos los resultados de las nuevas variables
df_features.select(
    "trip_distance",
    "trip_distance_km",
    "fare_amount",
    "fare_per_km"
).show(5)

+-------------+----------------+-----------+-----------+
|trip_distance|trip_distance_km|fare_amount|fare_per_km|
+-------------+----------------+-----------+-----------+
|         0.97|            1.56|        7.2|       4.62|
|         5.58|            8.98|       38.7|       4.31|
|         2.33|            3.75|       14.2|       3.79|
|          1.3|            2.09|       11.4|       5.45|
|         5.34|            8.59|       37.3|       4.34|
+-------------+----------------+-----------+-----------+
only showing top 5 rows


#**Exploratory Data Analysis with Spark**